# Naive Bayes Sentiment Classifier From Scratch

## 1. Project Goal

The goal of this notebook is to implement a simple **Multinomial Naive Bayes sentiment classifier** from scratch.

The model will learn from a small labeled dataset and classify new sentences as:

- `positive`
- `negative`
- `neutral/uncertain` when the scores are effectively tied

This notebook is intentionally small and educational. The goal is not to build a production-grade sentiment model, but to understand every component clearly.


In [ ]:
import re
from math import log


## 2. Bayes Theorem Refresher

Bayes theorem:

$$
P(A|B)=\frac{P(B|A)P(A)}{P(B)}
$$

In words:

> \(P(A|B)\) is the probability of event \(A\) after observing that event \(B\) has occurred.

For sentiment classification:

$$
P(positive|words)=\frac{P(words|positive)P(positive)}{P(words)}
$$

We compare:

$$
P(positive|words)
$$

against:

$$
P(negative|words)
$$

Both have the same denominator \(P(words)\), so for classification we only need to compare the numerators:

$$
P(words|positive)P(positive)
$$

vs

$$
P(words|negative)P(negative)
$$

This is called **proportional comparison**.


### Important terms

**Prior probability**

The probability of a class before looking at the sentence.

Example:

$$
P(positive)
$$

**Likelihood**

The probability of observing a word/sentence assuming a specific class.

Example:

$$
P(word|positive)
$$

**Posterior probability**

The updated probability after observing evidence.

Example:

$$
P(positive|words)
$$

In this notebook, we do not compute the exact posterior. We compute relative log-scores and choose the class with the larger score.


## 3. Dataset

Each training sample is a tuple:

```python
(sentence, label)
```

Label convention:

- `1` = positive
- `0` = negative


In [ ]:
train_data = [
    ("I love this movie", 1),
    ("This film is amazing", 1),
    ("What a great experience", 1),
    ("I really enjoyed this", 1),

    ("I hate this movie", 0),
    ("This film is terrible", 0),
    ("What a bad experience", 0),
    ("I really dislike this", 0),
]


In [ ]:
for sentence, label in train_data:
    print(f"Sentence: {sentence}")
    print(f"Label: {label}")
    print()


## 4. Text Preprocessing

The preprocessing pipeline here is intentionally simple:

1. Convert text to lowercase.
2. Remove punctuation and non-letter characters.
3. Split text into tokens.

This is enough for a small from-scratch Naive Bayes classifier.


In [ ]:
def preprocess(text):
    """
    Convert raw text into a list of normalized word tokens.

    Args:
        text (str): Raw input sentence.

    Returns:
        list[str]: Lowercased word tokens with punctuation removed.
    """
    text = text.lower()
    text = re.sub(r"[^a-zA-Z\s]", "", text)
    tokens = text.split()

    return tokens


In [ ]:
# Small test for preprocess()
assert preprocess("I LOVE this movie!!!") == ["i", "love", "this", "movie"]

preprocess("I LOVE this movie!!!")


## 5. Building Word Frequencies

Naive Bayes needs to know how often each word appears in each class.

We store this as:

```python
(word, label) -> count
```

Example:

```python
("love", 1) -> 3
("bad", 0) -> 2
```

A dictionary is the correct data structure because we need a mapping from a key to a frequency value.


In [ ]:
def build_freqs(train_data):
    """
    Build word-label frequency counts from the training data.

    Args:
        train_data (list[tuple[str, int]]): List of (sentence, label) pairs.

    Returns:
        dict[tuple[str, int], int]:
            A dictionary where:
            - key = (word, label)
            - value = number of times the word appeared in that label/class
    """
    freqs = {}

    for sentence, label in train_data:
        tokens = preprocess(sentence)

        for token in tokens:
            pair = (token, label)
            freqs[pair] = freqs.get(pair, 0) + 1

    return freqs


In [ ]:
# Small test for build_freqs()
sample_freqs = build_freqs([
    ("love love movie", 1),
    ("hate movie", 0),
])

assert sample_freqs[("love", 1)] == 2
assert sample_freqs[("movie", 1)] == 1
assert sample_freqs[("hate", 0)] == 1
assert sample_freqs[("movie", 0)] == 1

sample_freqs


## 6. Building the Vocabulary

The vocabulary is the set of all unique words seen during training.

A `set` is the correct data structure because vocabulary cares about uniqueness, not frequency.


In [ ]:
def build_vocab(freqs):
    """
    Build the vocabulary from a frequency dictionary.

    Args:
        freqs (dict[tuple[str, int], int]): Word-label frequency dictionary.

    Returns:
        set[str]: Set of unique words seen during training.
    """
    vocab = set()

    for word, label in freqs.keys():
        vocab.add(word)

    return vocab


In [ ]:
# Small test for build_vocab()
sample_vocab = build_vocab(sample_freqs)

assert sample_vocab == {"love", "movie", "hate"}

sample_vocab


## 7. Training Statistics

There are two different kinds of statistics we need.

### Document-level statistics: priors

These answer:

$$
P(positive)
$$

and:

$$
P(negative)
$$

They are based on how many training documents belong to each class.

### Word-level statistics: likelihood denominators

These answer how many total word occurrences appeared in each class.

They are used to compute:

$$
P(word|positive)
$$

and:

$$
P(word|negative)
$$

Important distinction:

```text
p_pos / p_neg = document-level probabilities
n_pos / n_neg = total word occurrences in each class
```


In [ ]:
def compute_priors(train_data):
    """
    Compute prior probabilities for the positive and negative classes.

    Args:
        train_data (list[tuple[str, int]]): List of (sentence, label) pairs.

    Returns:
        tuple[float, float]:
            p_pos: probability that a training document is positive.
            p_neg: probability that a training document is negative.
    """
    positive_count = 0
    negative_count = 0

    total_sentences = len(train_data)

    for sentence, label in train_data:
        if label == 1:
            positive_count += 1
        else:
            negative_count += 1

    p_pos = positive_count / total_sentences
    p_neg = negative_count / total_sentences

    return p_pos, p_neg


In [ ]:
# Small test for compute_priors()
p_pos, p_neg = compute_priors(train_data)

assert p_pos == 0.5
assert p_neg == 0.5

print("positive prior:", p_pos)
print("negative prior:", p_neg)


In [ ]:
def count_words_by_label(freqs):
    """
    Count total word occurrences in the positive and negative classes.

    Args:
        freqs (dict[tuple[str, int], int]): Word-label frequency dictionary.

    Returns:
        tuple[int, int]:
            n_pos: total word occurrences in positive training examples.
            n_neg: total word occurrences in negative training examples.
    """
    n_pos = 0
    n_neg = 0

    for (word, label), count in freqs.items():
        if label == 1:
            n_pos += count
        else:
            n_neg += count

    return n_pos, n_neg


In [ ]:
# Small test for count_words_by_label()
sample_n_pos, sample_n_neg = count_words_by_label(sample_freqs)

assert sample_n_pos == 3  # love love movie
assert sample_n_neg == 2  # hate movie

sample_n_pos, sample_n_neg


## 8. Laplace Smoothing

### Problem: zero probabilities

If a word never appeared in a class during training, then:

$$
P(word|class)=0
$$

When sentence probabilities are computed by multiplying word probabilities, a single zero probability makes the whole sentence probability zero.

### Solution: Laplace smoothing

$$
P(word|class)=
\frac{\text{word count}+1}
{\text{total words in class}+\text{vocabulary size}}
$$

The idea is to pretend that every word in the vocabulary has been observed once before.

- Adding `1` to the word count prevents zero probabilities.
- Adding the vocabulary size `V` to the denominator compensates for the extra counts added across all vocabulary words.


In [ ]:
def get_word_probability(freqs, word, label, n_pos, n_neg, V):
    """
    Compute the smoothed probability P(word | label).

    Args:
        freqs (dict[tuple[str, int], int]): Word-label frequency dictionary.
        word (str): Target word.
        label (int): Class label, where 1 = positive and 0 = negative.
        n_pos (int): Total positive word occurrences.
        n_neg (int): Total negative word occurrences.
        V (int): Vocabulary size.

    Returns:
        float: Laplace-smoothed probability of the word given the class.
    """
    word_count = freqs.get((word, label), 0)

    if label == 1:
        denominator = n_pos + V
    else:
        denominator = n_neg + V

    probability = (word_count + 1) / denominator

    return probability


In [ ]:
# Small test for get_word_probability()
controlled_freqs = {
    ("love", 1): 2,
    ("movie", 1): 3,
    ("great", 1): 1,
    ("hate", 0): 2,
    ("movie", 0): 1,
    ("bad", 0): 3,
}

controlled_vocab = {"love", "movie", "great", "hate", "bad"}
controlled_n_pos = 6
controlled_n_neg = 6
controlled_V = len(controlled_vocab)

assert get_word_probability(controlled_freqs, "love", 1, controlled_n_pos, controlled_n_neg, controlled_V) == 3 / 11
assert get_word_probability(controlled_freqs, "love", 0, controlled_n_pos, controlled_n_neg, controlled_V) == 1 / 11

print("P(love | positive):", get_word_probability(controlled_freqs, "love", 1, controlled_n_pos, controlled_n_neg, controlled_V))
print("P(love | negative):", get_word_probability(controlled_freqs, "love", 0, controlled_n_pos, controlled_n_neg, controlled_V))


## 9. Sentence Scoring

Naive Bayes assumes words are conditionally independent given the class:

$$
P(sentence|class)=P(w_1|class)P(w_2|class)\dots P(w_n|class)
$$

The full score is:

$$
P(class)P(sentence|class)
$$

However, multiplying many small probabilities can cause numerical underflow.

So we use logs:

$$
\log(a \times b)=\log(a)+\log(b)
$$

The sentence score becomes:

$$
\log(P(class))+\sum_i \log(P(w_i|class))
$$

The function below returns a **log-score for one class**, not an exact posterior probability.


In [ ]:
def score_sentence(sentence, label, freqs, vocab, n_pos, n_neg, p_pos, p_neg):
    """
    Compute the Naive Bayes log-score for a sentence under one class.

    Args:
        sentence (str): Input sentence to score.
        label (int): Class label to score against, where 1 = positive and 0 = negative.
        freqs (dict[tuple[str, int], int]): Word-label frequency dictionary.
        vocab (set[str]): Training vocabulary.
        n_pos (int): Total positive word occurrences.
        n_neg (int): Total negative word occurrences.
        p_pos (float): Positive prior probability.
        p_neg (float): Negative prior probability.

    Returns:
        float:
            Log-score for the sentence under the given class.
            Larger score means the sentence is more likely under that class.
    """
    if label == 1:
        score = log(p_pos)
    else:
        score = log(p_neg)

    tokens = preprocess(sentence)

    for token in tokens:
        # In this simple version, words unseen during training are ignored.
        if token in vocab:
            probability = get_word_probability(freqs, token, label, n_pos, n_neg, len(vocab))
            score += log(probability)

    return score


## 10. Training the Model

The training function collects all learned parameters into one object.

For this educational version, the model is a tuple:

```python
(freqs, vocab, n_pos, n_neg, p_pos, p_neg)
```


In [ ]:
def train_naive_bayes(train_data):
    """
    Train a simple Multinomial Naive Bayes sentiment classifier.

    Args:
        train_data (list[tuple[str, int]]): List of (sentence, label) pairs.

    Returns:
        tuple:
            freqs: word-label frequency dictionary.
            vocab: set of unique training words.
            n_pos: total positive word occurrences.
            n_neg: total negative word occurrences.
            p_pos: positive prior probability.
            p_neg: negative prior probability.
    """
    freqs = build_freqs(train_data)
    vocab = build_vocab(freqs)
    n_pos, n_neg = count_words_by_label(freqs)
    p_pos, p_neg = compute_priors(train_data)

    return freqs, vocab, n_pos, n_neg, p_pos, p_neg


In [ ]:
model = train_naive_bayes(train_data)
freqs, vocab, n_pos, n_neg, p_pos, p_neg = model

print("p_pos, p_neg:", p_pos, p_neg)
print("n_pos, n_neg:", n_pos, n_neg)
print("vocab size:", len(vocab))
print("vocab:", vocab)


In [ ]:
# Small training sanity checks
assert p_pos == 0.5
assert p_neg == 0.5
assert n_pos == 16
assert n_neg == 16
assert len(vocab) == 17


## 11. Prediction

Prediction works by:

1. Scoring the sentence as positive.
2. Scoring the sentence as negative.
3. Returning the class with the larger log-score.

Because log-probabilities are usually negative, the better score is the one closer to zero.

Example:

```python
-5 > -10
```

So `-5` is the larger score.


In [ ]:
def predict(sentence, model):
    """
    Predict the sentiment of a sentence using a trained Naive Bayes model.

    Args:
        sentence (str): Input sentence.
        model (tuple): Output of train_naive_bayes().

    Returns:
        tuple[str, float, float]:
            prediction: "positive", "negative", or "neutral/uncertain".
            positive_score: log-score for the positive class.
            negative_score: log-score for the negative class.
    """
    freqs, vocab, n_pos, n_neg, p_pos, p_neg = model

    positive_score = score_sentence(sentence, 1, freqs, vocab, n_pos, n_neg, p_pos, p_neg)
    negative_score = score_sentence(sentence, 0, freqs, vocab, n_pos, n_neg, p_pos, p_neg)

    if abs(positive_score - negative_score) < 1e-9:
        prediction = "neutral/uncertain"
    elif positive_score > negative_score:
        prediction = "positive"
    else:
        prediction = "negative"

    return prediction, positive_score, negative_score


In [ ]:
# Small tests for predict()
assert predict("I love this movie", model)[0] == "positive"
assert predict("This film is amazing", model)[0] == "positive"
assert predict("I hate this movie", model)[0] == "negative"
assert predict("This is terrible", model)[0] == "negative"

print(predict("I love this movie", model))
print(predict("This film is amazing", model))
print(predict("I hate this movie", model))
print(predict("This is terrible", model))


In [ ]:
# Mixed sentiment and short examples
print(predict("great movie", model))
print(predict("bad film", model))
print(predict("love and hate", model))
print(predict("fantastic movie", model))  # "fantastic" is unknown and ignored


## 12. Evaluation

For this small project, we use accuracy:

```text
accuracy = correct predictions / total predictions
```

This is enough for a tiny balanced dataset. For larger or imbalanced datasets, precision, recall, F1-score, and a confusion matrix would be more informative.


In [ ]:
def evaluate(test_data, model):
    """
    Evaluate the classifier using accuracy.

    Args:
        test_data (list[tuple[str, str]]):
            List of (sentence, true_label) pairs.
            true_label should be "positive" or "negative".

        model (tuple): Output of train_naive_bayes().

    Returns:
        float: Accuracy percentage.
    """
    total_sentences = len(test_data)
    correct_count = 0

    for sentence, true_label in test_data:
        prediction_label, positive_score, negative_score = predict(sentence, model)

        if prediction_label == true_label:
            correct_count += 1

    accuracy = (correct_count / total_sentences) * 100

    return accuracy


In [ ]:
test_data = [
    ("I love this film", "positive"),
    ("This movie is great", "positive"),
    ("I enjoyed this experience", "positive"),
    ("I hate this film", "negative"),
    ("This movie is bad", "negative"),
    ("This experience is terrible", "negative"),
]


In [ ]:
for sentence, true_label in test_data:
    prediction, positive_score, negative_score = predict(sentence, model)

    print(f"Sentence: {sentence}")
    print(f"True label: {true_label}")
    print(f"Prediction: {prediction}")
    print(f"Positive score: {positive_score:.4f}")
    print(f"Negative score: {negative_score:.4f}")
    print()


In [ ]:
accuracy = evaluate(test_data, model)
print("The accuracy of the model =", accuracy)

assert accuracy == 100.0


## 13. Limitations

This implementation is intentionally simple.

Main limitations:

- The dataset is tiny.
- Test examples are very similar to training examples.
- The model ignores word order.
- The model assumes words are conditionally independent given the class.
- Unknown words are ignored during prediction.
- The model uses a simple bag-of-words representation.
- Accuracy alone is not enough for serious evaluation on real data.


## 14. Final Pipeline

```text
train_data
    ↓
preprocess
    ↓
build_freqs
    ↓
build_vocab
    ↓
compute_priors
    ↓
count_words_by_label
    ↓
train_naive_bayes
    ↓
score_sentence
    ↓
predict
    ↓
evaluate
```

Final takeaway:

> Naive Bayes converts word counts into class probabilities, uses smoothing to avoid zero probabilities, uses log-scores to avoid numerical underflow, and predicts the class with the larger log-score.
